In [ ]:
# Import libraries
from pyspark.sql.functions import col, explode, isnull, when, to_timestamp, from_utc_timestamp, max, min, current_timestamp, to_date, hour
from delta import DeltaTable
import json

# Helper Notebook

Load helper notebook with the schema definition of silver table in "silver_table_schema" and merge keys in "silver_merge_keys".

In [ ]:
%run ./NB_Silver_Table_Schemas

In [ ]:
# Rename variables from helper notebook
target_table_schema = silver_table_schema
merge_keys = silver_merge_keys

# Parameters and Variables

In [ ]:
# Define parameters with None, as values will come from pipeline activities
# Source
source_lakehouse_path = None 
source_folder_name = None

# Target
target_lakehouse_path = None 
target_schema = None
target_table = None

# Dates
watermark_column = None
last_watermark = None

In [ ]:
# Define variables using values from parameters
source_path = f'{source_lakehouse_path}/Files/{source_folder_name}' 
target_path = f'{target_lakehouse_path}/Tables/{target_schema}/{target_table}'

# Read and Transform data from Bronze

In [ ]:
# Capture a fixed ingestion timestamp upfront to ensure batch consistency 
# and prevent timestamp drift if new files arrive in Bronze during processing.
ingestion_timestamp = current_timestamp()

In [ ]:
# Read files older than last watermark
df = (
    spark.read
        .option("recursiveFileLookup", "true")
        .option("modifiedAfter", last_watermark)
        .option("multiline", "true")
        .json(source_path)
)

In [ ]:
# Expand values inside atributtes of json file into rows and columns
df = (
    df
    .select(
        col("indicator.id").alias("indicator_id"),
        col("indicator.name").alias("indicator_name"),
        col("indicator.magnitud").getItem(0).getField("id").alias("magnitude_id"),
        col("indicator.magnitud").getItem(0).getField("name").alias("magnitude_name"),
        explode(col("indicator.values")).alias("value")
    )
    .select(
        "indicator_id",
        "indicator_name",
        "magnitude_id",
        "magnitude_name",
        col("value.datetime").alias("datetime_locale"),
        "value.datetime_utc",
        "value.geo_id",
        "value.geo_name",
        "value.value"
    )
)

In [ ]:
# Drop Duplicates
df = df.dropDuplicates(merge_keys)

In [ ]:
# Replace nulls with 0 in "value" column, cast date columns to datetime and split date and hour into new columns
df = (df
    .withColumn("datetime_locale", from_utc_timestamp(to_timestamp(col("datetime_locale")), "Europe/Madrid"))
    .withColumn("date_locale", to_date(col("datetime_locale")))
    .withColumn("hour_locale", hour(col("datetime_locale")))
    .withColumn("datetime_utc", to_timestamp(col("datetime_utc")))
    .withColumn('value', when(isnull(col('value')), 0).otherwise(col('value')))
)

In [ ]:
# Add ingestion_timestamp as new column
df = df.withColumn("ingestion_timestamp", ingestion_timestamp)

# Merge into Silver

In [ ]:
# Check if the table exists. If not, create it using the predefined schema
try:
    delta_table = DeltaTable.forPath(spark, target_path)
    print('Table already exists')
except:
    delta_table = DeltaTable.createIfNotExists(spark) \
    .location(target_path) \
    .addColumns(target_table_schema) \
    .execute()
    print('Table has been created')

In [ ]:
# Define the SQL merge condition using the column list (supports single or multiple columns)
# t: target; s: source
merge_condition = " AND ".join(f"t.{col} = s.{col}" for col in merge_keys) 

# Upsert 
delta_table = DeltaTable.forPath(spark, target_path)
delta_table.alias("t").merge(
    df.alias("s"),
    merge_condition
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Retrieve execution statistics
metrics = delta_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]
rows_inserted = int(metrics["numTargetRowsInserted"])
rows_updated = int(metrics["numTargetRowsUpdated"])

In [ ]:
# Obtain max date in watermark_column to use as new watermark and store it in a dictionary with the statistics
maxdate = df.select(max(watermark_column)).first()[0]

result = {
    "maxdate": maxdate.strftime("%Y-%m-%d %H:%M:%S") if maxdate else None,
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated
}

# Make the result values queriable for other activities in the pipeline
mssparkutils.notebook.exit(json.dumps(result))